In [ ]:
import pandas as pd

dataset_labels = {
    'ALS 2025-10-15 0.05s': 'als_20251015_50ms',
    'ALS 2025-10-15 0.1s': 'als_20251015_100ms',
}

ccstar_data = {}

# load all the merged data
for key, label in dataset_labels.items():
    for part in ['part1','part2']:
        ccstar_data[(key,part)] = pd.read_csv(f'source_data/fig5_cc_star_{label}_{part}.csv').set_index('s')

data = {}

# load the split merged data
for key, label in dataset_labels.items():
    data[key] = pd.read_csv(f'source_data/fig5_cc_{label}_part1_vs_part2.csv').set_index('s')

In [ ]:
import matplotlib.pyplot as plt

fig = plt.figure(figsize=(9, 4.5))
subfigs = fig.subfigures(nrows=1, ncols=2)

layout = """
AABB
.CC.
"""

for j, key in enumerate(dataset_labels):
    #fig, axs = plt.subplots(2, 2, sharex=True, sharey=True, figsize=(4.5,4.5))
    axs = subfigs[j].subplot_mosaic(layout, sharex=True, sharey=True)
    for part, axlabel in zip(['part1','part2'],['A','B']):
        ccstar_data[(key,part)]['cc_star'].plot(
            legend=False,
            color='tab:purple',
            ax=axs[axlabel])
        axs[axlabel].axhline(y=1, color='gray', linestyle='-', linewidth=0.5)
        axs[axlabel].set_title({'part1':'Part 1','part2':'Part 2'}[part])
        axs[axlabel].set_xlabel(None)
        axs[axlabel].set_ylim([0,1.1])
        axs[axlabel].set_ylabel('Correlation coefficient')
    data[key].plot(
        ax=axs['C'],
        legend=False,
        color=['tab:blue','tab:cyan'],
        style=['-','-'],
    )
    axs['C'].fill_between(data[key].index, data[key]["cc"], data[key]["cc_limit"], color="tab:cyan", alpha=0.2)
    axs['C'].set_xlabel('s (Å$^{-1}$)')
    axs['C'].axhline(y=1, color='gray', linestyle='-', linewidth=0.5)
    axs['C'].tick_params(labelbottom=True, labelleft=True)
    axs['C'].yaxis.label.set_visible(True)
    axs['C'].set_ylabel('Correlation coefficient')
    subfigs[j].suptitle(key.split()[-1], fontsize=14)
    subfigs[j].subplots_adjust(hspace=0.15, wspace=0.15) 

handles, labels = [], []
for axlabel in ['B','C']:
    h, l = axs[axlabel].get_legend_handles_labels()
    handles.extend(h)
    labels.extend(l)
# relabel
labels = [
    'CC*',
    'CC$_{pair}$',
    'CC$_{lim}$',
]
fig.legend(handles, labels, loc="lower center", ncols=1)

plt.savefig("fig5_cc.svg", transparent=True, bbox_inches="tight")
plt.savefig("fig5_cc.pdf", transparent=True, bbox_inches="tight")
plt.savefig("fig5_cc.png", transparent=True, bbox_inches="tight", dpi=300)